# diag_paris_sae_2 — which SAE features encode 'Paris'? (Llama-Scope / Neuronpedia)

Like `diag_paris_sae`, but uses the **Llama-Scope** SAEs (OpenMOSS/`fnlp`) instead of Goodfire — because their features have **public, clickable pages on Neuronpedia** (Goodfire's labels are behind the Ember API).

**Layers: 9, 17, 19** — your three current steering layers (depths 9/17/19 = indices -23/-15/-13). Not all layers (32 SAEs would be ~140 GB); these three load one-at-a-time.

**Model:** Llama-Scope was trained on the **base** model, so we run **`meta-llama/Llama-3.1-8B` (base)** — that keeps the JumpReLU thresholds / normalization valid and makes Neuronpedia's dashboards correspond. First run downloads the base model (~16 GB). To match your exact steering model, set `MODEL_ID` to the *Instruct* variant — but then the SAE is out-of-distribution and the feature sparsity may be off.

In [1]:
import torch, json, gc, math, os
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download
from safetensors import safe_open
os.makedirs('plots', exist_ok=True)

MODEL_ID = 'meta-llama/Llama-3.1-8B'        # base (matches the SAE). Instruct: 'meta-llama/Llama-3.1-8B-Instruct'
device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
dtype  = torch.float16 if device.type != 'cpu' else torch.float32
print('device:', device)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device).eval()
print('loaded', MODEL_ID)

device: mps


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

loaded meta-llama/Llama-3.1-8B


## Llama-Scope SAE loader (direct from safetensors — JumpReLU + dataset-wise input scaling)

In [2]:
SAE_REPO = 'fnlp/Llama3_1-8B-Base-LXR-32x'
LAYERS   = [9, 17, 19]                      # = your steering layers (depths 9/17/19)
def NP(L, fid):                             # public Neuronpedia feature page
    return f'https://www.neuronpedia.org/llama3.1-8b/{L}-llamascope-res-131k/{fid}'

def load_sae_encoder(L):
    base = f'Llama3_1-8B-Base-L{L}R-32x'
    hp = json.load(open(hf_hub_download(SAE_REPO, f'{base}/hyperparams.json')))
    fp = hf_hub_download(SAE_REPO, f'{base}/checkpoints/final.safetensors')
    with safe_open(fp, framework='pt') as f:           # load encoder only (saves ~2 GB)
        W_enc = f.get_tensor('encoder.weight').to(device).float()   # (131072, 4096)
        b_enc = f.get_tensor('encoder.bias').to(device).float()     # (131072,)
    thr   = float(hp['jump_relu_threshold'])
    scale = math.sqrt(hp['d_model']) / hp['dataset_average_activation_norm']['in']  # = norm_scaling_factor
    return W_enc, b_enc, thr, scale

def sae_features(x, W_enc, b_enc, thr, scale):
    pre = torch.nn.functional.linear(x.float() * scale, W_enc, b_enc)   # encode in normalized space
    return pre * (pre > thr)                                            # JumpReLU

## Grab the layer-L residual (blocks.L.hook_resid_post) at a target token

In [3]:
_buf = {}
def _grab(m, i, o): _buf['h'] = (o[0] if isinstance(o, tuple) else o).detach()
@torch.no_grad()
def residual_at(text, target, L):
    ids = tok(text, return_tensors='pt').to(device)
    h = model.model.layers[L].register_forward_hook(_grab); model(**ids); h.remove()
    strs = [tok.decode([t]) for t in ids.input_ids[0]]
    pos = max(j for j, s in enumerate(strs) if target.lower() in s.lower())
    return _buf['h'][0, pos, :], strs, pos

## Top SAE features for 'Paris' at each of L9 / L17 / L19
Each feature prints a clickable Neuronpedia link — open it to read the human label + max-activating examples.

In [4]:
PROMPT, TARGET = 'The Eiffel Tower is located in Paris.', 'Paris'
paris_feats = {}
for L in LAYERS:
    W_enc, b_enc, thr, scale = load_sae_encoder(L)
    x, strs, pos = residual_at(PROMPT, TARGET, L)
    f = sae_features(x, W_enc, b_enc, thr, scale)
    paris_feats[L] = f.detach().clone()
    top = torch.topk(f, 15)
    print(f'\n===== Layer {L}  —  token {strs[pos]!r}  —  {(f>0).sum().item()} active features =====')
    for v, idx in zip(top.values.tolist(), top.indices.tolist()):
        print(f'  feat {idx:6d}  act={v:6.2f}   {NP(L, idx)}')
    del W_enc, b_enc; gc.collect()
    if device.type == 'mps': torch.mps.empty_cache()


===== Layer 9  —  token ' Paris'  —  140 active features =====
  feat 129752  act= 26.08   https://www.neuronpedia.org/llama3.1-8b/9-llamascope-res-131k/129752
  feat  55862  act= 15.98   https://www.neuronpedia.org/llama3.1-8b/9-llamascope-res-131k/55862
  feat 114577  act= 11.25   https://www.neuronpedia.org/llama3.1-8b/9-llamascope-res-131k/114577
  feat  97218  act=  8.20   https://www.neuronpedia.org/llama3.1-8b/9-llamascope-res-131k/97218
  feat 104724  act=  7.83   https://www.neuronpedia.org/llama3.1-8b/9-llamascope-res-131k/104724
  feat   6914  act=  7.36   https://www.neuronpedia.org/llama3.1-8b/9-llamascope-res-131k/6914
  feat 110086  act=  6.18   https://www.neuronpedia.org/llama3.1-8b/9-llamascope-res-131k/110086
  feat    158  act=  5.04   https://www.neuronpedia.org/llama3.1-8b/9-llamascope-res-131k/158
  feat 109035  act=  5.00   https://www.neuronpedia.org/llama3.1-8b/9-llamascope-res-131k/109035
  feat  64504  act=  4.50   https://www.neuronpedia.org/llama3.1-8b/9-

hyperparams.json: 0.00B [00:00, ?B/s]

Llama3_1-8B-Base-L17R-32x/checkpoints/fi(…):   0%|          | 0.00/2.15G [00:00<?, ?B/s]


===== Layer 17  —  token ' Paris'  —  111 active features =====
  feat  81357  act= 17.68   https://www.neuronpedia.org/llama3.1-8b/17-llamascope-res-131k/81357
  feat  87235  act= 13.04   https://www.neuronpedia.org/llama3.1-8b/17-llamascope-res-131k/87235
  feat 119595  act= 12.79   https://www.neuronpedia.org/llama3.1-8b/17-llamascope-res-131k/119595
  feat  27390  act= 10.70   https://www.neuronpedia.org/llama3.1-8b/17-llamascope-res-131k/27390
  feat  68076  act=  8.09   https://www.neuronpedia.org/llama3.1-8b/17-llamascope-res-131k/68076
  feat  65714  act=  7.24   https://www.neuronpedia.org/llama3.1-8b/17-llamascope-res-131k/65714
  feat  68656  act=  6.58   https://www.neuronpedia.org/llama3.1-8b/17-llamascope-res-131k/68656
  feat  58605  act=  6.33   https://www.neuronpedia.org/llama3.1-8b/17-llamascope-res-131k/58605
  feat   6351  act=  5.34   https://www.neuronpedia.org/llama3.1-8b/17-llamascope-res-131k/6351
  feat  24998  act=  4.46   https://www.neuronpedia.org/llama3

hyperparams.json: 0.00B [00:00, ?B/s]

Llama3_1-8B-Base-L19R-32x/checkpoints/fi(…):   0%|          | 0.00/2.15G [00:00<?, ?B/s]


===== Layer 19  —  token ' Paris'  —  84 active features =====
  feat  34467  act= 26.20   https://www.neuronpedia.org/llama3.1-8b/19-llamascope-res-131k/34467
  feat 107730  act= 11.73   https://www.neuronpedia.org/llama3.1-8b/19-llamascope-res-131k/107730
  feat  11267  act= 10.70   https://www.neuronpedia.org/llama3.1-8b/19-llamascope-res-131k/11267
  feat 107026  act=  8.46   https://www.neuronpedia.org/llama3.1-8b/19-llamascope-res-131k/107026
  feat 106067  act=  8.17   https://www.neuronpedia.org/llama3.1-8b/19-llamascope-res-131k/106067
  feat 126295  act=  6.71   https://www.neuronpedia.org/llama3.1-8b/19-llamascope-res-131k/126295
  feat  25776  act=  6.29   https://www.neuronpedia.org/llama3.1-8b/19-llamascope-res-131k/25776
  feat 117168  act=  6.25   https://www.neuronpedia.org/llama3.1-8b/19-llamascope-res-131k/117168
  feat  72783  act=  6.07   https://www.neuronpedia.org/llama3.1-8b/19-llamascope-res-131k/72783
  feat  85364  act=  5.45   https://www.neuronpedia.org/ll

## Paris-specific features (Paris vs London contrast, per layer)
Features much stronger for Paris than London isolate city identity from shared 'city/capital' features.

In [5]:
for L in LAYERS:
    W_enc, b_enc, thr, scale = load_sae_encoder(L)
    xP, *_ = residual_at('The Eiffel Tower is located in Paris.', 'Paris', L)
    xL, *_ = residual_at('Big Ben is located in London.', 'London', L)
    fP = sae_features(xP, W_enc, b_enc, thr, scale)
    fL = sae_features(xL, W_enc, b_enc, thr, scale)
    diff = torch.topk(fP - fL, 10)
    print(f'\n===== Layer {L}: stronger for Paris than London =====')
    for v, idx in zip(diff.values.tolist(), diff.indices.tolist()):
        print(f'  feat {idx:6d}  Paris={fP[idx]:6.2f}  London={fL[idx]:6.2f}  (d {v:+.2f})  {NP(L, idx)}')
    del W_enc, b_enc; gc.collect()
    if device.type == 'mps': torch.mps.empty_cache()


===== Layer 9: stronger for Paris than London =====
  feat  55862  Paris= 15.98  London= -0.00  (d +15.98)  https://www.neuronpedia.org/llama3.1-8b/9-llamascope-res-131k/55862
  feat  64504  Paris=  4.50  London= -0.00  (d +4.50)  https://www.neuronpedia.org/llama3.1-8b/9-llamascope-res-131k/64504
  feat 105620  Paris=  4.03  London= -0.00  (d +4.03)  https://www.neuronpedia.org/llama3.1-8b/9-llamascope-res-131k/105620
  feat 103136  Paris=  4.08  London=  1.01  (d +3.07)  https://www.neuronpedia.org/llama3.1-8b/9-llamascope-res-131k/103136
  feat  29184  Paris=  3.88  London=  1.43  (d +2.45)  https://www.neuronpedia.org/llama3.1-8b/9-llamascope-res-131k/29184
  feat  80355  Paris=  3.96  London=  1.65  (d +2.31)  https://www.neuronpedia.org/llama3.1-8b/9-llamascope-res-131k/80355
  feat   7633  Paris=  2.15  London= -0.00  (d +2.15)  https://www.neuronpedia.org/llama3.1-8b/9-llamascope-res-131k/7633
  feat   9642  Paris=  1.99  London= -0.00  (d +1.99)  https://www.neuronpedia.org/l

## Reading this
- Open any feature's Neuronpedia link for its label + max-activating examples (e.g. 'France/French', 'European capitals', or a dedicated 'Paris' feature).
- A clean dedicated Paris (or Paris/France) feature at L9/L17/L19 means the model **holds Paris as a discrete feature right where you steer** -> the steering failure is about *extraction/geometry*, not absence.
- The Paris-London contrast separates city-specific features from shared city/capital ones.
- Note: Llama-Scope is base-model-trained; feature IDs/labels here are faithful on the **base** model (`MODEL_ID` default). Switching to Instruct keeps the same feature IDs but activations go slightly OOD.